In [2]:
import pandas as pd

In [3]:
# Read all kpler data excel files from a folder and concatenate them into a single dataframe, there are dozens of files, so we need to read them in a loop and concatenate them.
import os

folder_path = '../data/kpler_trade_flow_data/'
all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.xlsx')]

dfs = []
for f in all_files:
    try:
        dfs.append(pd.read_excel(f, engine='openpyxl'))
    except Exception as e:
        print(f"Skipped {f}: {e}")

kpler_data = pd.concat(dfs, ignore_index=True)
kpler_data.shape

(22262, 178)

In [ ]:
# kpler_data.to_excel('../data/Processed_data/kpler_trade_flow_data_raw.xlsx', index=False)

In [5]:
# Count how many unique sellers-buyers pairs, we have sellers and buyers in separate columns, we want to count the unique pairs of sellers and buyers.
kpler_data['seller_buyer_pair'] = kpler_data['Seller (origin)'] + ' - ' + kpler_data['Buyer (destination)']
unique_pairs = kpler_data['seller_buyer_pair'].nunique()
print(f'The number of unique seller-buyer pairs is: {unique_pairs}')    
print(f"The number of unique sellers is: {kpler_data['Seller (origin)'].nunique()}")
print(f"The number of unique buyers is: {kpler_data['Buyer (destination)'].nunique()}")


# Output the unique sellers-port pairs
kpler_data['seller_port_pair'] = kpler_data['Seller (origin)'] + ' - ' + kpler_data['Installation origin']
unique_seller_port_pairs = kpler_data['seller_port_pair'].nunique()
print(f'The number of unique seller-port pairs is: {unique_seller_port_pairs}')

# Get kpler dataframe without blank values in Seller (origin), Buyer (Destination), Installation origin, Installation Destination, and Product, and count the unique seller-port pairs again
kpler_data_no_unknown = kpler_data.dropna(subset=['Seller (origin)', 'Buyer (destination)', 'Installation origin', 'Installation Destination', 'Product'])
print(kpler_data_no_unknown.shape)

# print unique seller-installation origin-buyer-installation destination-product pairs, for those with blank, we will fill them with "Unknown"
kpler_data_no_unknown['seller_buyer_product_pair'] = kpler_data_no_unknown['Seller (origin)'] + ' - ' + \
    kpler_data_no_unknown['Installation origin'] + ' - ' + kpler_data_no_unknown['Buyer (destination)']\
        + ' - ' + kpler_data_no_unknown['Installation Destination'] + ' - ' + kpler_data_no_unknown['Product']
unique_seller_buyer_product_pairs = kpler_data_no_unknown['seller_buyer_product_pair'].nunique()
print(f'The number of unique seller-installation origin-buyer-installation destination-product pairs is: {unique_seller_buyer_product_pairs}')

The number of unique seller-buyer pairs is: 1018
The number of unique sellers is: 208
The number of unique buyers is: 318
The number of unique seller-port pairs is: 240
(6815, 180)
The number of unique seller-installation origin-buyer-installation destination-product pairs is: 1992


C:\Users\JXW997\AppData\Local\Temp\ipykernel_42884\3136869432.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  kpler_data['seller_buyer_pair'] = kpler_data['Seller (origin)'] + ' - ' + kpler_data['Buyer (destination)']
C:\Users\JXW997\AppData\Local\Temp\ipykernel_42884\3136869432.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  kpler_data['seller_port_pair'] = kpler_data['Seller (origin)'] + ' - ' + kpler_data['Installation origin']
C:\Users\JXW997\AppData\Local\Temp\ipykernel_42884\3136869432.py:19: PerformanceWarning: D

In [37]:
kpler_data_no_unknown.head()

,Vessel,Country (origin),Zone Origin,Installation origin,End (origin),Country (destination),Zone Destination,Installation Destination,Start (destination),Cargo (tons),...,Forecasted origins confidence,Forecasted destinations,Forecasted destinations confidence,Bill Number,Shipper,Consignee,Notify Party,seller_buyer_pair,seller_port_pair,seller_buyer_product_pair
5,Herbert C Jackson,United States,Marquette,LS&I Ore Dock,2025-04-06 22:05,United States,Detroit,Cleveland Cliffs Dearborn Works,2025-04-09 09:42,5950,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cleveland-Cliffs - AK Steel,Cleveland-Cliffs - LS&I Ore Dock,Cleveland-Cliffs - LS&I Ore Dock - AK Steel - ...
9,Feng Hua,Australia,Port Hedland,Finucane Island,2025-04-06 20:35,China,Luojing,Taicang Ore,2025-04-25 04:53,86749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BHP - Shagang Group,BHP - Finucane Island,BHP - Finucane Island - Shagang Group - Taican...
10,Feng Hua,Australia,Port Hedland,Finucane Island,2025-04-06 20:35,China,Rizhao Port,Rizhao Dry Bulks,2025-04-21 03:49,86749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BHP - Shagang Group,BHP - Finucane Island,BHP - Finucane Island - Shagang Group - Rizhao...
17,Amnsi Stallion,India,Paradip,Paradip Central Quay Terminal,2025-04-06 18:13,India,Hazira,AM/NS Hazira,2025-04-16 12:14,78712,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AM/NS - AM/NS,AM/NS - Paradip Central Quay Terminal,AM/NS - Paradip Central Quay Terminal - AM/NS ...
21,Star Tembu,Australia,Cape Preston,Sino Iron,2025-04-06 12:53,China,Ningbo,Ningbo Steel Plant,2025-04-23 06:35,55545,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Citic Pacific Mining - Baosteel,Citic Pacific Mining - Sino Iron,Citic Pacific Mining - Sino Iron - Baosteel - ...


In [ ]:
# # Aggregate trade flow by seller-buyer-product pairs, and sum the Cargo (tons), output the entire agregated dataframe to a new excel file, need to keep other columns as well
# agg_dict = {col: 'first' for col in kpler_data.columns if col != 'seller_buyer_product_pair'}
# agg_dict['Cargo (tons)'] = 'sum'
# # agg_df should keep the column of seller_buyer_product_pair
# agg_df = kpler_data_no_unknown.groupby('seller_buyer_product_pair').agg(agg_dict).reset_index()
# agg_df.to_excel('../data/Processed_data/kpler_aggregated_trade_flow.xlsx', index=False)

In [ ]:
# Aggregate trade flow by seller-buyer-product pairs, and sum the Cargo (tons), output the entire agregated dataframe to a new excel file, need to keep other columns as well
agg_dict = {col: 'first' for col in kpler_data_no_unknown.columns if col != 'seller_buyer_product_pair'}
agg_dict['Cargo (tons)'] = 'sum'
# agg_df should keep the column of seller_buyer_product_pair
agg_df = kpler_data_no_unknown.groupby('seller_buyer_product_pair').agg(agg_dict).reset_index()

# For agg_df, separate the sellers inthe Seller (origin) column by comma and explode the dataframe so that each row only has one seller, and do the same for buyers in the Buyer (destination) column, and keep other columns the same. Noted that we need to keep all sellers rather than just the first seller. For the cargo (tons) column, we need to average the original cargo (tons) for the same seller-buyer-product pair, rather than just taking the first value, because after we explode the sellers and buyers, we will have multiple rows for the same seller-buyer-product pair, and we want to keep the total cargo (tons) for that pair. After exploding, output the entire dataframe to a new excel file.
# Count combinations before exploding to split cargo correctly
agg_df['_n_sellers'] = agg_df['Seller (origin)'].str.split(',').str.len().fillna(1)
agg_df['_n_buyers']  = agg_df['Buyer (destination)'].str.split(',').str.len().fillna(1)
agg_df['_n_combos']  = agg_df['_n_sellers'] * agg_df['_n_buyers']

# Split into lists
agg_df_exploded = agg_df.assign(
    **{'Seller (origin)':    agg_df['Seller (origin)'].str.split(','),
       'Buyer (destination)': agg_df['Buyer (destination)'].str.split(',')}
)

# Explode separately → cross product (each seller paired with each buyer)
agg_df_exploded = agg_df_exploded.explode('Seller (origin)', ignore_index=True)
agg_df_exploded = agg_df_exploded.explode('Buyer (destination)', ignore_index=True)

# Divide cargo by number of seller-buyer combos
agg_df_exploded['Cargo (tons)'] = agg_df_exploded['Cargo (tons)'] / agg_df_exploded['_n_combos']

# Cleanup
agg_df_exploded['Seller (origin)']    = agg_df_exploded['Seller (origin)'].str.strip()
agg_df_exploded['Buyer (destination)'] = agg_df_exploded['Buyer (destination)'].str.strip()
agg_df_exploded = agg_df_exploded.drop(columns=['_n_sellers', '_n_buyers', '_n_combos'])

agg_df_exploded.to_excel('../data/Processed_data/kpler_aggregated_trade_flow_exploded.xlsx', index=False)


In [56]:
# Reasoning the iron ore mine plant based on the columns including 'Seller (origin)', 'Installation origin', 'Zone Origin' and 'Country (origin)'
extracted_seller_data = agg_df_exploded[['Seller (origin)', 'Installation origin', 'Zone Origin', 'Country (origin)']]
extracted_seller_data.to_excel('../data/Processed_data/kpler_extracted_seller_data.xlsx', index=False)
# extracted_buyer_data = agg_df_exploded[['Buyer (destination)', 'Installation Destination', 'Zone Destination', 'Country (destination)']]
# extracted_buyer_data.to_excel('../data/Processed_data/kpler_extracted_buyer_data.xlsx', index=False)


In [29]:
# Infer operator_city for seller and buyer
def data_melt(df):
    city_cols = [f'city{i}' for i in range(1, 11)]

    # Everything except the city columns stays per-row and is repeated across melted rows
    id_cols = [c for c in df.columns if c not in city_cols]

    # Stack each city column into its own block of rows
    frames = []
    for i in range(1, 11):
        part = df[id_cols].copy()
        part['city'] = df[f'city{i}']
        part['pair_idx'] = i
        frames.append(part)

    long_df = pd.concat(frames, ignore_index=True)

    # Drop rows where city is blank/NaN
    def is_blank(s):
        return s.isna() | (s.astype(str).str.strip().isin(['', 'nan', 'None']))

    long_df = (long_df[~is_blank(long_df['city'])]
               .sort_values(id_cols + ['pair_idx'])
               .reset_index(drop=True))

    return long_df


seller_df = pd.read_excel('../data/kpler_trade_flow_data/processed_data/kpler_seller_city_mine_operator.xlsx', sheet_name='Seller Mine-Plant Inference', skiprows=2)
buyer_df = pd.read_excel('../data/kpler_trade_flow_data/processed_data/buyer_inferences.xlsx', sheet_name='Buyer Inferences')
seller_df = seller_df.iloc[:,:-15]
buyer_df = buyer_df.iloc[:,:-11]
seller_melt = data_melt(seller_df)
buyer_melt = data_melt(buyer_df)
# concentrate the operator and city columns into a single column, and drop the pair_idx column
seller_melt['seller_operator_city'] = seller_melt['correct_operator'] + '_' + seller_melt['city']
seller_melt = seller_melt.drop(columns=['pair_idx'])
buyer_melt['buyer_operator_city'] = buyer_melt['correct_operator'] + '_' + buyer_melt['city']
buyer_melt = buyer_melt.drop(columns=['pair_idx'])

In [31]:
# seller_melt.to_excel('../data/kpler_trade_flow_data/processed_data/seller_melt.xlsx', index=False)
# buyer_melt.to_excel('../data/kpler_trade_flow_data/processed_data/buyer_melt.xlsx', index=False)

In [49]:
def join_operator_city(kpler_trade_flow, seller_melt, buyer_melt):
    kpler_trade_flow = kpler_trade_flow[~kpler_trade_flow['Product'].isin(
        ['Intermediate Iron Products', 'Bentonite', 'Magnetite', 'Aluminium'])].copy()
    kpler_trade_flow['Product_type'] = kpler_trade_flow['Product'].str.replace('Iron Ore ', '', regex=False)

    # 1) tag each ORIGINAL shipment BEFORE any join, and stash the true tonnage
    kpler_trade_flow = kpler_trade_flow.reset_index(drop=True)
    kpler_trade_flow['ship_id'] = kpler_trade_flow.index
    original_total = kpler_trade_flow['Cargo (tons)'].sum()   # for the check at the end

    seller_key = ['Seller (origin)', 'Installation origin', 'Zone Origin', 'Country (origin)']
    buyer_key  = ['Buyer (destination)', 'Installation Destination', 'Zone Destination', 'Country (destination)']

    # 2) de-duplicate the right tables on their keys so each side fans out at most as intended.
    #    (protects against accidental extra dupes in the lookup tables)
    seller_lu = seller_melt[seller_key + ['seller_operator_city']].drop_duplicates()
    buyer_lu  = buyer_melt[buyer_key + ['buyer_operator_city']].drop_duplicates()

    # 3) both left joins — pairing stays intact, rows may duplicate
    df = (kpler_trade_flow
          .merge(seller_lu, on=seller_key, how='left')
          .merge(buyer_lu,  on=buyer_key,  how='left'))

    # 4) split ONCE by the total number of rows each shipment became
    df['split_weight'] = 1 / df.groupby('ship_id')['ship_id'].transform('size')
    df['Cargo (tons)'] = df['Cargo (tons)'] * df['split_weight']

    # 5) verify the invariant
    assert abs(df['Cargo (tons)'].sum() - original_total) < 1e-6, "tonnage not conserved!"

    # 6）remove rows with either seller_operator_city or buyer_operator_city missing, because we cannot infer the operator_city for those rows
    df = df.dropna(subset=['seller_operator_city', 'buyer_operator_city'])

    return df


kpler_trade_flow = pd.read_excel('../data/kpler_trade_flow_data/processed_data/kpler_aggregated_trade_flow_exploded.xlsx', sheet_name='Sheet1')
kpler_trade_flow_df = join_operator_city(kpler_trade_flow, seller_melt, buyer_melt)
kpler_trade_flow_df.to_excel('../data/kpler_trade_flow_data/processed_data/kpler_trade_flow_with_operator_city_pair.xlsx', index=False)


In [ ]:
df1 = pd.read_excel('../data/Processed_data/steel_property_cost_merged.xlsx', sheet_name='Sheet1')
df2 = pd.read_excel('../data/Processed_data/iron_mine_w_cost_FeContent.xlsx', sheet_name='Sheet1')


(1794, 186)